# TED Talks Content Recommender

I wanted a quick recommender that, given the title of a TED talk I liked, suggests a few others I might also like. No collaborative signal here — just TF-IDF on the descriptions and tags, then cosine similarity. The whole thing is small enough to fit in memory.

I tried to use a Kaggle TED talks CSV, but I'm running this without internet on the train ride home, so I generated a small synthetic stand-in of ~200 talks. The code reads `../../data/ted_talks/ted_talks.csv` if it exists, otherwise it builds the synthetic one. When I plug the real dataset in the logic stays the same.

In [1]:
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(7)
np.random.seed(7)

DATA = Path('../../data/ted_talks')
DATA.mkdir(parents=True, exist_ok=True)
CSV = DATA / 'ted_talks.csv'

In [2]:
# Synth-talks fallback. The Kaggle TED dataset (`rounakbanik/ted-talks`) has
# title, description, tags, main_speaker, etc. I'm only using the first three.
if not CSV.exists():
    THEMES = {
        'creativity': ['art', 'design', 'imagination', 'storytelling', 'music'],
        'science':    ['physics', 'biology', 'space', 'research', 'evolution'],
        'tech':       ['ai', 'robotics', 'internet', 'software', 'startups'],
        'society':    ['politics', 'inequality', 'culture', 'history', 'identity'],
        'health':     ['medicine', 'mental health', 'nutrition', 'aging', 'sleep'],
        'climate':    ['climate', 'energy', 'oceans', 'sustainability', 'conservation'],
        'education':  ['learning', 'schools', 'teachers', 'curiosity', 'children'],
    }
    OPENERS = [
        'A surprising look at',
        'What we get wrong about',
        'The hidden story behind',
        'Lessons from a decade studying',
        'Rethinking the future of',
        'Why I changed my mind on',
    ]
    rows = []
    for i in range(200):
        theme = random.choice(list(THEMES))
        tags = random.sample(THEMES[theme], k=3)
        opener = random.choice(OPENERS)
        title = f'{opener} {tags[0]}'
        desc = (f'In this talk the speaker explores {tags[0]}, '
                f'drawing on years of work in {tags[1]} and {tags[2]}. '
                f'The argument centers on how {theme} shapes the way we think.')
        rows.append({'talk_id': i, 'title': title, 'tags': ' '.join(tags), 'description': desc})
    pd.DataFrame(rows).to_csv(CSV, index=False)
    print('synthesized', len(rows), 'talks ->', CSV)
else:
    print('using existing', CSV)

synthesized 200 talks -> ../../data/ted_talks/ted_talks.csv


In [3]:
df = pd.read_csv(CSV)
print(df.shape)
df.head(3)

(200, 4)


,talk_id,title,tags,description
0,0,A surprising look at robotics,robotics software internet,"In this talk the speaker explores robotics, dr..."
1,1,Rethinking the future of music,music art design,"In this talk the speaker explores music, drawi..."
2,2,A surprising look at music,music design art,"In this talk the speaker explores music, drawi..."


## Build the TF-IDF feature matrix

I'm concatenating title, tags, and description into a single text blob per talk. Tags get repeated twice on purpose so the vectorizer weights them a bit more than prose.

In [4]:
df['blob'] = (df['title'].fillna('') + ' ' +
              (df['tags'].fillna('') + ' ') * 2 +
              df['description'].fillna(''))

vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, max_df=0.8)
X = vec.fit_transform(df['blob'])
print('tfidf shape:', X.shape, ' vocab size:', len(vec.vocabulary_))

tfidf shape: (200, 459)  vocab size: 459


In [5]:
sim = cosine_similarity(X)
# diagonal is 1.0 by definition; zero it out so a talk doesn't recommend itself
np.fill_diagonal(sim, 0.0)
sim.shape

(200, 200)

## Recommend

Given the index of a seed talk, return the top-k most similar. I added a small diversity tweak — if two candidates share the exact same tag string I skip the second. Otherwise the synthetic set's top-5 was almost always five clones.

In [6]:
def recommend(seed_idx, k=5, dedup_tags=True):
    scores = sim[seed_idx]
    order = np.argsort(-scores)
    out, seen_tags = [], set()
    for j in order:
        if dedup_tags and df.loc[j, 'tags'] in seen_tags:
            continue
        seen_tags.add(df.loc[j, 'tags'])
        out.append((j, float(scores[j])))
        if len(out) == k:
            break
    return out

seed = 0
print('SEED ->', df.loc[seed, 'title'], '|', df.loc[seed, 'tags'])
print()
for j, s in recommend(seed, k=5):
    print(f'  {s:.3f}  {df.loc[j, "title"]}  ({df.loc[j, "tags"]})')

SEED -> A surprising look at robotics | robotics software internet

  0.966  What we get wrong about robotics  (robotics software internet)
  0.591  A surprising look at startups  (startups software internet)
  0.546  Lessons from a decade studying robotics  (robotics software startups)
  0.479  A surprising look at robotics  (robotics internet ai)
  0.477  Rethinking the future of robotics  (robotics ai internet)


## Notes

- Cosine similarity on TF-IDF is a baseline. Sentence embeddings (e.g. MiniLM) would almost certainly do better at the description level, but the tags carry most of the signal here anyway.
- The cold-start problem doesn't really exist for content-based methods — every new talk is recommendable as soon as it's in the index — but the flip side is that it can't pick up things only humans would notice (a great speaker, a memorable Q&A).
- I'd swap the dedup-by-tags hack for MMR if I were taking this further.